# IRF3 Nuclear Intensity Distribution

Segments nuclei from DAPI fluorescence images and quantifies IRF3 nuclear intensity distribution by comparing mean IRF3 fluorescence inside each nucleus to a surrounding cytoplasmic ring. Each nucleus is classified as nuclear or non-nuclear. Results are exported to CSV for downstream statistical analysis.

## 1. Dependencies

In [ ]:
import os
import cv2
import numpy as np
import pandas as pd

## 2. Configuration

In [ ]:
# --- Input ---
directory = "image2_npy"  # Input folder; each .npy file is a (2, H, W) array: [IRF3, DAPI]

# --- Group mapping ---
group_map = {
    '19': 'Mock_24H',
    '20': 'GS_24H',
    '22': 'GS_32H',
    '24': 'GS_48H'
}
order = ['19', '20', '22', '24']

# --- Nucleus segmentation parameters ---
min_area = 3500  # Minimum valid nucleus area (pixels)
min_circularity = 0.65  # Shape filter to exclude elongated or fragmented objects (established range: 0.6–0.7)
global_blur_size = (55, 55)    # Gaussian blur size for DAPI before thresholding
specific_blur_size = (65, 65)  # Blur within each ROI to refine nuclear mask
rescue_threshold = -20         # %Diff cutoff for "rescued" nuclear calls

# --- Morphology ring radii (~pixels) ---
inner_dilate = 11  # Inner erosion radius; excludes the 10 px immediately adjacent to the nuclear boundary
outer_dilate = 31  # Outer dilation radius; net cytoplasmic ring is ~20 px (outer 30 px minus excluded inner 10 px)

## 3. Image Processing Pipeline

In [ ]:
# ===============================================================
# INITIALIZE STORAGE
# ===============================================================
delta_by_group = {k: [] for k in group_map}
results_by_group = {k: [] for k in group_map}

# ===============================================================
# MAIN LOOP
# ===============================================================
npy_files = sorted([f for f in os.listdir(directory) if f.endswith(".npy")])
all_areas = []
circularities = []

for filename in npy_files:
    prefix = filename.split("_")[0]
    if prefix not in group_map:
        continue

    path = os.path.join(directory, filename)
    data = np.load(path)

    if data.shape[0] < 2:
        print(f"[warn] {filename} missing channels, skipping")
        continue

    irf3_gray = data[0].astype(np.uint8)
    dapi_gray = data[1].astype(np.uint8)
    height, width = dapi_gray.shape

    # --- Global nucleus segmentation on DAPI ---
    blurred = cv2.GaussianBlur(dapi_gray, global_blur_size, 0)
    _, thresh = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    areas = [cv2.contourArea(cnt) for cnt in contours]
    all_areas.extend(areas)

    def is_fully_inside(cnt):
        x, y, w, h = cv2.boundingRect(cnt)
        return x > 0 and y > 0 and (x + w) < width and (y + h) < height

    filtered_contours = [
        c for c in contours if cv2.contourArea(c) >= min_area and is_fully_inside(c)
    ]

    # --- Per-nucleus analysis ---
    for j, cnt in enumerate(filtered_contours):
        x, y, w, h = cv2.boundingRect(cnt)
        pad_x, pad_y = int(w * 0.25), int(h * 0.25)
        x1, y1 = max(x - pad_x, 0), max(y - pad_y, 0)
        x2, y2 = min(x + w + pad_x, width), min(y + h + pad_y, height)

        roi_irf3 = irf3_gray[y1:y2, x1:x2]
        roi_dapi = dapi_gray[y1:y2, x1:x2]

        # Secondary blur refines the nuclear boundary within the cropped ROI
        roi_blurred = cv2.GaussianBlur(roi_dapi, specific_blur_size, 0)
        _, roi_thresh = cv2.threshold(roi_blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        roi_contours, _ = cv2.findContours(roi_thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if not roi_contours:
            continue

        largest_cnt = max(roi_contours, key=cv2.contourArea)
        area = cv2.contourArea(largest_cnt)
        perimeter = cv2.arcLength(largest_cnt, True)
        if perimeter == 0:
            continue

        circularity = 4 * np.pi * (area / (perimeter ** 2))
        circularities.append(circularity)
        if area < min_area or circularity < min_circularity:
            continue

        mask_inside = np.zeros(roi_dapi.shape, dtype=np.uint8)
        cv2.drawContours(mask_inside, [largest_cnt], -1, 255, -1)

        # Create cytoplasmic ring to compare inside of boundary to outside of boundary
        kernel_inner = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (inner_dilate, inner_dilate))
        kernel_outer = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (outer_dilate, outer_dilate))
        dilated_inner = cv2.dilate(mask_inside, kernel_inner, iterations=1)
        dilated_outer = cv2.dilate(mask_inside, kernel_outer, iterations=1)
        ring_mask = cv2.subtract(dilated_outer, dilated_inner)

        # IRF3 intensity metrics
        background_mask = (roi_irf3 > 10).astype(np.uint8) * 255
        cytoring_mask = cv2.bitwise_and(ring_mask, background_mask)

        mean_inside = cv2.mean(roi_irf3, mask=mask_inside)[0]
        mean_cytoring = cv2.mean(roi_irf3, mask=cytoring_mask)[0]
        if mean_cytoring == 0:
            continue

        delta = mean_inside - mean_cytoring
        percent_diff = 100 * delta / mean_cytoring

        label = f"{filename}_cell{j}"
        delta_by_group[prefix].append(delta)
        results_by_group[prefix].append(
            (label, delta, percent_diff, roi_irf3.copy(), roi_dapi.copy(), largest_cnt.copy())
        )

## 4. Export Results

In [ ]:
# ===============================================================
# Save Data
# ===============================================================
all_rows = []

for group_key in order:
    condition = group_map[group_key]
    for (label, delta, percent_diff, roi_irf3, roi_dapi, largest_cnt) in results_by_group[group_key]:
        # Pre-rescue and rescued classifications
        classification_pre = "nuclear" if delta > 0 else "non-nuclear"
        classification_rescued = "nuclear" if percent_diff >= rescue_threshold else "non-nuclear"

        # Compute nucleus area and mean IRF3 (density)
        if roi_irf3 is not None and largest_cnt is not None:
            mask = np.zeros(roi_irf3.shape, dtype=np.uint8)
            cv2.drawContours(mask, [largest_cnt], -1, 255, -1)
            area = int(np.count_nonzero(mask))
            mean_irf3 = float(cv2.mean(roi_irf3, mask=mask)[0]) if area > 0 else np.nan
        else:
            area, mean_irf3 = np.nan, np.nan

        all_rows.append({
            "label": label,
            "condition": condition,
            "percent_diff": percent_diff,
            "classification": classification_rescued,
            "area": area,
            "mean_irf3": mean_irf3
        })

df = pd.DataFrame(all_rows)
df.to_csv("image2_results.csv", index=False)